In [1]:
pip install ollama

Defaulting to user installation because normal site-packages is not writeable
  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
Using cached ollama-0.6.1-py3-none-any.whl (14 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import ollama 

response = ollama.generate(
    model="deepseek-v3.1:671b-cloud",
    prompt="Why do stars twinkle?, write in 2000 words, and include a poem at the end."
)
print(response["response"])

Of all the celestial phenomena that have captivated humanity since the dawn of consciousness, few are as universally accessible and enchanting as the simple twinkling of a star. That gentle, rhythmic flickering, shifting between soft white and subtle hues, has inspired poets, guided navigators, and sparked the curiosity of scientists for millennia. It is a dance of light played out over unimaginable distances, a beautiful illusion born from the turbulent interface between the serene vacuum of space and the dynamic, living envelope of our own planet. The scientific explanation for this twinkling, known to astronomers as *scintillation*, is a fascinating story of physics, perception, and planetary science.

To understand why stars twinkle, we must first appreciate the nature of starlight itself. A star is a colossal furnace of nuclear fusion, emitting a steady, unwavering stream of light photons across the entire electromagnetic spectrum. By the time this light traverses the mind-bogglin

In [ ]:
pip install langchain_ollama langchain_chroma langchain_community langchain_text_splitters langsmith python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
# 2. Write code for document/pdf loader with LangSmith tracing + debug
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langsmith import traceable, Client

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# Load .env into notebook kernel environment (explicitly)
env_path = find_dotenv(filename=".env", usecwd=True)
print(f"[DEBUG] .env found: {bool(env_path)} -> {env_path if env_path else 'NOT FOUND'}")
if env_path:
    load_dotenv(dotenv_path=env_path, override=True)
else:
    # Fallback: try current working directory
    cwd_env = Path.cwd() / ".env"
    print(f"[DEBUG] Trying cwd .env: {cwd_env}")
    if cwd_env.exists():
        load_dotenv(dotenv_path=cwd_env, override=True)

# Enable tracing explicitly
os.environ["LANGSMITH_TRACING"] = "true"
os.environ.setdefault("LANGSMITH_PROJECT", "ollama-rag-debug")

# Debug checks
required_env = ["LANGSMITH_API_KEY", "LANGSMITH_ENDPOINT", "LANGSMITH_PROJECT", "LANGSMITH_TRACING"]
for key in required_env:
    value = os.getenv(key, "")
    print(f"[DEBUG] {key} set: {bool(value)}")

endpoint = os.getenv("LANGSMITH_ENDPOINT", "")
print(f"[DEBUG] LANGSMITH_ENDPOINT looks valid: {endpoint.startswith('http')}")

try:
    _client = Client()
    print("[DEBUG] LangSmith client initialized")
except Exception as e:
    print(f"[DEBUG] LangSmith client init failed: {e}")

@traceable(name="load_pdf")
def load_pdf(pdf_path: str):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    print(f"[DEBUG] Loaded {len(docs)} pages from {pdf_path}")
    return docs

@traceable(name="split_documents")
def split_documents(docs, chunk_size: int = 1000, chunk_overlap: int = 50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)
    print(f"[DEBUG] Created {len(chunks)} chunks")
    return chunks

@traceable(name="setup_pipeline")
def setup_pipeline(pdf_path: str):
    model = ChatOllama(model="llama3.2:1b", temperature=0)
    docs = load_pdf(pdf_path)
    chunks = split_documents(docs)

    embeddings = OllamaEmbeddings(model="nomic-embed-text")  # or "mxbai-embed-large"
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

    prompt_template = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a witty assistant who answers in a paragraph of 8 sentences. "
            "Use the following context to inform your answer: {context}. "
            "If you do not know the answer, say you do not know. "
            "Always use the context to answer the question."
        ),
        ("human", "{user_input}")
    ])

    parser = StrOutputParser()
    chain = (
        RunnableParallel({
            "context": (lambda x: x["user_input"]) | retriever,
            "user_input": (lambda x: x["user_input"])
        })
        | prompt_template
        | model
        | parser
    )
    print("[DEBUG] Pipeline setup complete")
    return chain, retriever

@traceable(name="smoke_test_trace")
def smoke_test_trace():
    return "trace-ok"

# Build pipeline
chain, retriever = setup_pipeline("money.pdf")

# Execution
response = chain.invoke({"user_input": "what are the habits will make u rich according to the author?, write down in 20 words"})
print(response)
print("[DEBUG] Smoke trace result:", smoke_test_trace())

[DEBUG] .env found: True -> c:\Users\C-TARA MTDS 3\Desktop\New projects\Ollama\.env
[DEBUG] LANGSMITH_API_KEY set: True
[DEBUG] LANGSMITH_ENDPOINT set: True
[DEBUG] LANGSMITH_PROJECT set: True
[DEBUG] LANGSMITH_TRACING set: True
[DEBUG] LANGSMITH_ENDPOINT looks valid: True
[DEBUG] LangSmith client initialized
[DEBUG] Loaded 242 pages from money.pdf
[DEBUG] Created 408 chunks
[DEBUG] Pipeline setup complete
The author suggests that habits like saving, investing, and living below your means can lead to wealth accumulation.
[DEBUG] Smoke trace result: trace-ok


In [13]:
retrieved_docs = retriever.invoke("what are the habits will make u rich according to the author?")
print("\n--- Retrieved Documents ---")
for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1}:\n{doc.page_content}\n---")
    #print(f"Count: {i+1}")


--- Retrieved Documents ---
Document 1:
But wealth is hidden. It’s income not spent. Wealth is an option not yet
taken to buy something later. Its value lies in offering you options,
flexibility, and growth to one day purchase more stuff than you could right
now.
Diet and exercise offer a useful analogy. Losing weight is notoriously hard,
even among those putting in the work of vigorous exercise. In his book The
Body, Bill Bryson explains why:
 
One study in America found that people overestimate the number of
calories they burned in a workout by a factor of four. They also then
consumed, on average, about twice as many calories as they had just burned
off … the fact is, you can quickly undo a lot of exercise by eating a lot of
food, and most of us do.
 
Exercise is like being rich. You think, “I did the work and I now deserve to
treat myself to a big meal.” Wealth is turning down that treat meal and
actually burning net calories. It’s hard, and requires self-control. But it
---
Docum

In [20]:
pip install ollama pyttsx3

Defaulting to user installation because normal site-packages is not writeable

   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ---------------------------------------- 3/3 [pyttsx3]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import pyttsx3
import ollama

def generate_and_speak():
    # 1. Initialize the free, offline text-to-speech engine
    engine = pyttsx3.init()

    # Optional: Customize the voice (speed and volume)
    engine.setProperty('rate', 170)  # Speed of speech (words per minute)
    engine.setProperty('volume', 1.0) # Volume (0.0 to 1.0)

    # 3. Convert the generated text to speech and play it
    print("Speaking...")
    engine.say(response)
    engine.runAndWait()

    # 4. Save the voiceover directly to an audio file (Great for video editing workflows)
    engine.save_to_file(response, 'final_voiceover.mp3')
    engine.runAndWait()
    print("Saved audio to final_voiceover.mp3")

if __name__ == "__main__":
    generate_and_speak()

Speaking...
Saved audio to final_voiceover.mp3


In [30]:
import pyttsx3
import ollama

def generate_and_speak():
    # 1. Initialize the free, offline text-to-speech engine
    engine = pyttsx3.init()



    # --- CHANGE VOLUME ---
    # Ranges from 0.0 (silent) to 1.0 (loudest)
    engine.setProperty('volume', 0.9) 

    # --- FIND AND CHANGE VOICES / LANGUAGES ---
    voices = engine.getProperty('voices')

    # This will print out all the voices currently installed on your OS
    print("Available Voices:")
    for index, voice in enumerate(voices):
        print(f"[{index}] Name: {voice.name} | Languages: {voice.languages}")

        # Swap the '1' below with the index number of the voice you want to use
        # Typically, index 0 is male and index 1 is female on Windows
        engine.setProperty('voice', voices[1].id) 

        # Test it out
        engine.say(response)
        engine.runAndWait()

if __name__ == "__main__":
    generate_and_speak()

Available Voices:
[0] Name: Microsoft David Desktop - English (United States) | Languages: ['en-US']
[1] Name: Microsoft Zira Desktop - English (United States) | Languages: ['en-US']


In [29]:
import pyttsx3

def test_all_voices():
    # 1. Initialize the engine
    engine = pyttsx3.init()
    engine.setProperty('volume', 0.9) 

    # 2. Get all available voices on your system
    voices = engine.getProperty('voices')

    print(f"Found {len(voices)} available voices.")
    print("Testing each voice now...\n")

    # 3. Loop through and listen to every single one
    for index, voice in enumerate(voices):
        print(f"Testing Voice [{index}] - Name: {voice.name}")
        
        # THE FIX: Set the engine to the current voice's ID in the loop
        engine.setProperty('voice', voice.id) 
        
        # Create a test phrase so the voice announces its own number
        test_phrase = f"Hello, this is a test. I am voice number {index}."
        
        # Test it out
        engine.say(response)
        engine.runAndWait()

if __name__ == "__main__":
    test_all_voices()

Found 2 available voices.
Testing each voice now...

Testing Voice [0] - Name: Microsoft David Desktop - English (United States)
Testing Voice [1] - Name: Microsoft Zira Desktop - English (United States)
